# Windows Class Tutorial

## About the Windows Class

The `Windows` class is the **core component** of the rolling_windows module. It's responsible for taking your text and dividing it into overlapping segments (windows) that can then be analyzed by calculators.

Think of it as a **text segmentation tool** - like taking a long scroll and cutting it into overlapping pieces so you can examine each piece individually while maintaining context from adjacent sections.

> **Tip:** Before diving in, check out the project README for an overview of all components (within the whole module and this class) and how they fit together. This will help you understand the broader workflow and make the most of the Windows class.

## Prerequisites

In [1]:
import spacy
from lexos.rolling_windows import Windows
from pathlib import Path

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

## Understanding Window Creation

### Basic Initialization

In [2]:
# Create a Windows instance
windows = Windows()

# Check default settings
print(f"Default window size: {windows.n}")
print(f"Default window type: {windows.window_type}")  
print(f"Default alignment mode: {windows.alignment_mode}")
print(f"Default output format: {windows.output}")

Default window size: 1000
Default window type: characters
Default alignment mode: strict
Default output format: strings


### Custom Initialization

In [3]:
# Create windows with custom settings
custom_windows = Windows(
    n=50,                    # Window size
    window_type="tokens",    # Type of units
    alignment_mode="strict", # How to handle boundaries
    output="strings"         # Output format
)

print("Custom settings applied!")

Custom settings applied!


## Input Types Supported

The Windows class accepts various input formats:

### 1. Raw Text Strings

In [4]:
sample_text = "This is a sample text for window analysis. It demonstrates how text gets segmented."

# Create character-based windows
char_windows = windows(
    input=sample_text,
    n=20,  # 20 characters per window
    window_type="characters",
    output="strings"
)

print("Character windows:")
for i, window in enumerate(list(char_windows)[:3]):
    print(f"Window {i+1}: '{window}'")

Character windows:
Window 1: 'This is a sample tex'
Window 2: 'his is a sample text'
Window 3: 'is is a sample text '


### 2. spaCy Doc Objects

In [5]:
# Process text with spaCy first
doc = nlp("The quick brown fox jumps over the lazy dog. This sentence follows.")

# Create token-based windows
token_windows = windows(
    input=doc,
    n=5,  # 5 tokens per window
    window_type="tokens", 
    output="strings"
)

print("\nToken windows:")
for i, window in enumerate(list(token_windows)[:3]):
    print(f"Window {i+1}: {window}")


Token windows:
Window 1: The quick brown fox jumps
Window 2: quick brown fox jumps over
Window 3: brown fox jumps over the


### 3. Lists of Strings

In [6]:
# Pre-tokenized text as list
token_list = ["The", "quick", "brown", "fox", "jumps", "over", "lazy", "dog"]

list_windows = windows(
    input=token_list,
    n=3,  # 3 tokens per window
    window_type="tokens",
    output="strings"
)

print("\nList-based windows:")
for i, window in enumerate(list(list_windows)):
    print(f"Window {i+1}: {window}")


List-based windows:
Window 1: ['The', 'quick', 'brown']
Window 2: ['quick', 'brown', 'fox']
Window 3: ['brown', 'fox', 'jumps']
Window 4: ['fox', 'jumps', 'over']
Window 5: ['jumps', 'over', 'lazy']
Window 6: ['over', 'lazy', 'dog']


### 4. spaCy Token Lists

In [7]:
# Extract tokens from spaCy doc
tokens = [token for token in doc]

spacy_windows = windows(
    input=tokens,
    n=4,
    window_type="tokens",
    output="tokens"  # Keep as spaCy tokens
)

print("\nspaCy token windows:")
for i, window in enumerate(list(spacy_windows)[:2]):
    token_texts = [token.text for token in window]
    pos_tags = [token.pos_ for token in window]
    print(f"Window {i+1}: {token_texts}")
    print(f"POS tags: {pos_tags}")


spaCy token windows:
Window 1: ['The', 'quick', 'brown', 'fox']
POS tags: ['DET', 'ADJ', 'ADJ', 'NOUN']
Window 2: ['quick', 'brown', 'fox', 'jumps']
POS tags: ['ADJ', 'ADJ', 'NOUN', 'VERB']


## Window Types Explained

### Characters Windows
Best for analyzing punctuation, symbols, or character-level patterns:


In [8]:
text = "Hello, world! How are you today?"

char_windows = windows(
    input=text,
    n=10,
    window_type="characters",
    output="strings"
)

print("Character windows (10 chars each):")
for i, window in enumerate(list(char_windows)[:4]):
    print(f"'{window}'")

Character windows (10 chars each):
'Hello, wor'
'ello, worl'
'llo, world'
'lo, world!'


### Token Windows  
Most common choice for word-level analysis:

In [9]:
token_windows = windows(
    input=doc,
    n=6,
    window_type="tokens",
    output="strings"  
)

print("\nToken windows (6 tokens each):")
for i, window in enumerate(list(token_windows)[:3]):
    print(f"Window {i+1}: {window}")


Token windows (6 tokens each):
Window 1: The quick brown fox jumps over
Window 2: quick brown fox jumps over the
Window 3: brown fox jumps over the lazy


### Span Windows
For advanced linguistic analysis with spaCy spans:

In [10]:
# Create spans from sentences
spans = [sent for sent in doc.sents]

span_windows = windows(
    input=spans,
    n=1,  # One sentence per window
    window_type="spans",
    output="strings"
)

print("\nSpan windows (sentence-based):")
for i, window in enumerate(list(span_windows)):
    print(f"Sentence {i+1}: {window}")


Span windows (sentence-based):
Sentence 1: ['The quick brown fox jumps over the lazy dog.']
Sentence 2: ['This sentence follows.']


## Output Formats

### String Output (Human-Readable)

In [11]:
string_output = windows(
    input=doc[:10],  # First 10 tokens
    n=4,
    window_type="tokens",
    output="strings"  # Human-readable text
)

print("String output:")
for i, window in enumerate(list(string_output)):
    print(f"Window {i+1}: {window}")

String output:
Window 1: The quick brown fox
Window 2: quick brown fox jumps
Window 3: brown fox jumps over
Window 4: fox jumps over the
Window 5: jumps over the lazy
Window 6: over the lazy dog
Window 7: the lazy dog.


### Token Output (Preserves Linguistic Info)

In [12]:
token_output = windows(
    input=doc[:10], 
    n=4,
    window_type="tokens",
    output="tokens"  # spaCy token objects
)

print("\nToken output (with linguistic features):")
for i, window in enumerate(list(token_output)):
    for token in window:
        print(f"  {token.text} -> POS: {token.pos_}, Lemma: {token.lemma_}")
    print("---")


Token output (with linguistic features):
  The -> POS: DET, Lemma: the
  quick -> POS: ADJ, Lemma: quick
  brown -> POS: ADJ, Lemma: brown
  fox -> POS: NOUN, Lemma: fox
---
  quick -> POS: ADJ, Lemma: quick
  brown -> POS: ADJ, Lemma: brown
  fox -> POS: NOUN, Lemma: fox
  jumps -> POS: VERB, Lemma: jump
---
  brown -> POS: ADJ, Lemma: brown
  fox -> POS: NOUN, Lemma: fox
  jumps -> POS: VERB, Lemma: jump
  over -> POS: ADP, Lemma: over
---
  fox -> POS: NOUN, Lemma: fox
  jumps -> POS: VERB, Lemma: jump
  over -> POS: ADP, Lemma: over
  the -> POS: DET, Lemma: the
---
  jumps -> POS: VERB, Lemma: jump
  over -> POS: ADP, Lemma: over
  the -> POS: DET, Lemma: the
  lazy -> POS: ADJ, Lemma: lazy
---
  over -> POS: ADP, Lemma: over
  the -> POS: DET, Lemma: the
  lazy -> POS: ADJ, Lemma: lazy
  dog -> POS: NOUN, Lemma: dog
---
  the -> POS: DET, Lemma: the
  lazy -> POS: ADJ, Lemma: lazy
  dog -> POS: NOUN, Lemma: dog
  . -> POS: PUNCT, Lemma: .
---


## Alignment Modes

Alignment modes control how character-based windows align with token boundaries:

### Strict Mode (Default)
Only creates windows that respect token boundaries:

In [13]:
text = "Hello world test"
doc = nlp(text)

strict_windows = windows(
    input=doc,
    n=8,  # 8 characters
    window_type="characters", 
    alignment_mode="strict",
    output="strings"
)

print("Strict alignment:")
for window in list(strict_windows):
    print(f"'{window}'")

Strict alignment:


### Contract Mode
Contracts windows to fit within token boundaries:

In [14]:
contract_windows = windows(
    input=doc,
    n=8,
    window_type="characters",
    alignment_mode="contract", 
    output="strings"
)

print("\nContract alignment:")
for window in list(contract_windows):
    print(f"'{window}'")


Contract alignment:


### Expand Mode  
Expands windows to include complete tokens:

In [15]:
expand_windows = windows(
    input=doc,
    n=8,
    window_type="characters",
    alignment_mode="expand",
    output="strings"
)

print("\nExpand alignment:")
for window in list(expand_windows):
    print(f"'{window}'")


Expand alignment:


## Working with Real Text Files

In [16]:
# Load the Beowulf test file (if available)
try:
    with open("BeowulfFullLines.txt", "r", encoding="utf-8") as f:
        beowulf_text = f.read()
    
    # Clean text (lowercase, remove extra spaces)
    clean_text = " ".join(beowulf_text.lower().split())
    beowulf_doc = nlp(clean_text[:1000])  # First 1000 characters
    
    # Create windows for analysis
    analysis_windows = windows(
        input=beowulf_doc,
        n=50,  # 50 tokens per window
        window_type="tokens",
        output="strings"
    )
    
    print(f"Created {len(list(analysis_windows))} windows from Beowulf text")
    
except FileNotFoundError:
    print("BeowulfFullLines.txt not found - using sample text instead")
    
    # Use sample Old English text
    sample_oe = "Hwæt! We Gardena in geardagum þeodcyninga þrym gefrunon"
    sample_doc = nlp(sample_oe)
    
    sample_windows = windows(
        input=sample_doc,
        n=5,
        window_type="tokens", 
        output="strings"
    )
    
    print("Sample Old English windows:")
    for i, window in enumerate(list(sample_windows)):
        print(f"Window {i+1}: {window}")

Created 7 windows from Beowulf text


## Common Patterns and Best Practices

### 1. Generator Consumption Warning

In [17]:
# Windows are generators - they get consumed!
demo_windows = windows(input="Test text for demonstration", n=5, window_type="characters")

# First use
first_list = list(demo_windows)
print(f"First consumption: {len(first_list)} windows")

# Second use - empty!
second_list = list(demo_windows) 
print(f"Second consumption: {len(second_list)} windows")

# Solution: Create fresh windows for each use
fresh_windows = windows(input="Test text for demonstration", n=5, window_type="characters")
print(f"Fresh windows: {len(list(fresh_windows))} windows")

First consumption: 23 windows
Second consumption: 0 windows
Fresh windows: 23 windows


### 2. Memory-Efficient Processing

In [18]:
# Process windows without storing all in memory
large_text = "This is a very long text. " * 100  # Simulate large document
large_doc = nlp(large_text)

large_windows = windows(
    input=large_doc,
    n=20,
    window_type="tokens",
    output="strings"
)

# Process one window at a time
window_count = 0
for window in large_windows:
    # Do analysis on individual window
    word_count = len(window.split())
    if window_count < 3:  # Show first 3 examples
        print(f"Window {window_count + 1}: {word_count} words")
    window_count += 1

print(f"Total windows processed: {window_count}")

Window 1: 5 words
Window 2: 4 words
Window 3: 5 words
Total windows processed: 104


### 3. Window Size Guidelines

In [19]:
sample_text = "A short sample text for testing different window sizes and their effects."
sample_doc = nlp(sample_text)

# Test different window sizes
sizes = [3, 5, 10, 15]

for size in sizes:
    test_windows = windows(
        input=sample_doc,
        n=size,
        window_type="tokens",
        output="strings"
    )
    
    window_list = list(test_windows)
    print(f"Window size {size}: {len(window_list)} windows generated")
    if window_list:
        print(f"  Example: '{window_list[0]}'")

Window size 3: 0 windows generated
Window size 5: 1 windows generated
  Example: 'short'
Window size 10: 0 windows generated
Window size 15: 0 windows generated


## Error Handling and Validation

In [21]:
# Test various edge cases
try:
    # Invalid window size
    bad_windows = Windows(n=0)
except ValueError as e:
    print(f"Invalid window size error: {e}")

try:
    # Invalid window type  
    bad_type = windows(input="test", window_type="invalid")
except Exception as e:
    print(f"Invalid window type error: {e}")

try:
    # Invalid output format (use valid window_type)
    bad_output = windows(input="test", window_type="tokens", output="invalid")
except Exception as e:
    print(f"Invalid output format error: {e}")

Invalid window size error: 1 validation error for Windows
n
  Input should be greater than 0 [type=greater_than, input_value=0, input_type=int]
    For further information visit https://errors.pydantic.dev/2.10/v/greater_than
Invalid window type error: Window type must be 'characters' or 'tokens'.
Invalid output format error: Output must be 'strings' or 'tokens'.


## Summary

The Windows class is your **text segmentation engine**:

- **Input flexibility**: Handles strings, spaCy docs, token lists, span lists
- **Window types**: Characters, tokens, or spans
- **Output formats**: Strings (readable) or tokens (with linguistic data)  
- **Alignment control**: Strict, contract, or expand modes
- **Memory efficient**: Generator-based processing

**Key takeaway**: Always create fresh Windows instances for each analysis since generators get consumed after use!

## Next Steps

Once you have windows, use them with:
- **Calculators** (`Counts`, `Averages`, `Ratios`) to analyze patterns
- **Plotters** (`SimplePlotter`, `PlotlyPlotter`) to visualize results